<a href="https://colab.research.google.com/github/Kanoa-Krohn/Quantum-Hackathon-Alpha-Squadron/blob/main/fianl-q-hackathon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Variational Quantum Eigensolver for a CO2 Molecule

## Environment Setup

In [72]:
%pip install scipy qiskit pyscf qiskit_nature qiskit_algorithms

from IPython.display import clear_output
clear_output()

print("All dependencies successfully installed!")

All dependencies successfully installed!


## Define Molecule Geometry
- Initialize the $\text{CO}_2$ molecule structure using PySCF with the desired basis set.

In [73]:
from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.drivers import PySCFDriver

driver = PySCFDriver(
    atom="O 0 0 -1.1621; C 0 0 0; O 0 0 1.1621",
    basis="631g",
    charge=0,
    spin=0,
    unit=DistanceUnit.ANGSTROM,
)

es_problem = driver.run()

## Apply Active Space Reduction
- Reduce the system to 4 active electrons and 4 active orbitals, then generate the fermionic Hamiltonian.

In [74]:
from qiskit_nature.second_q.transformers import ActiveSpaceTransformer

as_transformer = ActiveSpaceTransformer(4, 4)
es_problem = as_transformer.transform(es_problem)

fermionic_op = es_problem.hamiltonian.second_q_op()

## Display System Parameters
- Print the particle count, orbital count, and orbital energies prior to qubit mapping.

In [75]:
print("Before transformation:")
print("Number of particles:", es_problem.num_particles)
print("Number of orbitals:", es_problem.num_spatial_orbitals)
print("\nOrbital energies:",es_problem.orbital_energies)

Before transformation:
Number of particles: (2, 2)
Number of orbitals: 4

Orbital energies: [-0.53754586 -0.53754586  0.17998428  0.17998428]


## Map Fermions to Qubits
- Transform the fermionic Hamiltonian into qubit operators using Jordan-Wigner and Parity mappers.

In [76]:
from qiskit_nature.second_q.mappers import JordanWignerMapper, ParityMapper

jw_mapper = JordanWignerMapper()
qubit_jw_op = jw_mapper.map(fermionic_op)

p_mapper = ParityMapper(num_particles=es_problem.num_particles)
qubit_op = p_mapper.map(fermionic_op)
print(f"Num Qubits: {qubit_op.num_qubits}")
print(f"Num Terms:  {len(qubit_op)}")

Num Qubits: 6
Num Terms:  141


## Configure VQE & UCCSD Ansatz
- Build the Hartree-Fock initial state, define the UCCSD ansatz, and set up the VQE solver with the SLSQP optimizer.

In [77]:
from qiskit_algorithms import VQE
from qiskit_algorithms.optimizers import SLSQP
from qiskit.primitives import StatevectorEstimator
from qiskit_nature.second_q.circuit.library import HartreeFock, UCCSD

ansatz = UCCSD(
    es_problem.num_spatial_orbitals,
    es_problem.num_particles,
    p_mapper,
    initial_state=HartreeFock(
        es_problem.num_spatial_orbitals,
        es_problem.num_particles,
        p_mapper,
    ),
)

vqe_solver = VQE(StatevectorEstimator(), ansatz, SLSQP())
vqe_solver.initial_point = [0.0] * ansatz.num_parameters

## Run VQE Ground State Calculation
- Execute the VQE ground state solver and print total energy, particle numbers, and dipole moments.

In [78]:
from qiskit_nature.second_q.algorithms import GroundStateEigensolver

calc_vqe = GroundStateEigensolver(p_mapper, vqe_solver)
res_vqe = calc_vqe.solve(es_problem)
print(res_vqe)

/usr/local/lib/python3.13/dist-packages/scipy/sparse/linalg/_dsolve/linsolve.py:606: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/usr/local/lib/python3.13/dist-packages/scipy/sparse/linalg/_matfuncs.py:707: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)


=== GROUND STATE ENERGY ===
 
* Electronic ground state energy (Hartree): -245.845658105351
  - computed part:      -4.987195914643
  - ActiveSpaceTransformer extracted energy part: -240.858462190709
~ Nuclear repulsion energy (Hartree): 58.286449529094
> Total ground state energy (Hartree): -187.559208576258
 
=== MEASURED OBSERVABLES ===
 
  0:  # Particles: 4.000 S: 0.000 S^2: 0.000 M: 0.000
 
=== DIPOLE MOMENTS ===
 
~ Nuclear dipole moment (a.u.): [0.0  0.0  0.0]
 
  0: 
  * Electronic dipole moment (a.u.): [0.0  0.0  -0.000000404908]
    - computed part:      [0.0  0.0  -0.000000404908]
    - ActiveSpaceTransformer extracted energy part: [0.0  0.0  0.0]
  > Dipole moment (a.u.): [0.0  0.0  0.000000404908]  Total: 0.000000404908
                 (debye): [0.0  0.0  0.000001029174]  Total: 0.000001029174
 


## Calculate Exact Benchmark Energy
- Compute the exact ground state energy using NumPyMinimumEigensolver for validation.

In [79]:
from qiskit_algorithms import VQE, NumPyMinimumEigensolver
from qiskit_nature.second_q.algorithms import GroundStateEigensolver

numpy_solver = NumPyMinimumEigensolver()
calc_numpy = GroundStateEigensolver(p_mapper, numpy_solver)
res_numpy = calc_numpy.solve(es_problem)
print(res_numpy)

=== GROUND STATE ENERGY ===
 
* Electronic ground state energy (Hartree): -245.845675388827
  - computed part:      -4.987213198118
  - ActiveSpaceTransformer extracted energy part: -240.858462190709
~ Nuclear repulsion energy (Hartree): 58.286449529094
> Total ground state energy (Hartree): -187.559225859733
 
=== MEASURED OBSERVABLES ===
 
  0:  # Particles: 4.000 S: 0.000 S^2: 0.000 M: 0.000
 
=== DIPOLE MOMENTS ===
 
~ Nuclear dipole moment (a.u.): [0.0  0.0  0.0]
 
  0: 
  * Electronic dipole moment (a.u.): [0.0  0.0  0.0]
    - computed part:      [0.0  0.0  0.0]
    - ActiveSpaceTransformer extracted energy part: [0.0  0.0  0.0]
  > Dipole moment (a.u.): [0.0  0.0  0.0]  Total: 0.0
                 (debye): [0.0  0.0  0.0]  Total: 0.0
 


## Compare Results

In [85]:
print(f"VQE Energy: {res_vqe.total_energies[0]}")
print(f"Numpy Energy: {res_numpy.total_energies[0]}")

print(f"Difference: {res_vqe.total_energies[0] - res_numpy.total_energies[0]}")

VQE Energy: -187.5592085762575
Numpy Energy: -187.55922585973278
Difference: 1.7283475273188742e-05
